# Data preprocessing and duplicate review

Run with **Python (coms6998)** inside a Slurm compute allocation. No GPU is needed.

`scripts/prepare_data.py` owns downloading, hashing, grouping and splitting. This notebook reviews its audit; it does not modify source images or manifests. Run the cells in order, then change the filter arguments to browse candidates.

Decide whether images show the **same underlying image** (possibly resized, recompressed or adjusted), are **distinct**, or remain **uncertain**. Similar anatomy, orientation or class alone is insufficient. Inspect accepted and rejected pairs, especially near the correlation cutoff and across labels. Do not use classification results to select thresholds or claim patient-disjoint splits without patient IDs.

In [ ]:
import base64
import csv
import hashlib
import html
import json
import os
from pathlib import Path
import platform
import tempfile

from IPython.display import HTML, display
from PIL import Image, ImageOps
import io

if platform.node().startswith(("2402-login-", "insomnia-login")):
    raise RuntimeError("Open this notebook on a Slurm compute node, not the login node.")

# Find the repository even when the kernel starts in notebooks/.
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/data.json").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Start Jupyter within the brain-mri-vlm checkout.")
CONFIG = json.loads((REPO / "configs/data.json").read_text())

def repo_path(value):
    path = Path(value).expanduser()
    return (path if path.is_absolute() else REPO / path).resolve()

# Change these two paths if you use another audit directory or download cache.
AUDIT_DIR = repo_path(os.environ.get("MRI_DATA_OUTPUT", "data/prepared-v2-phash4-ncc98"))
cache = repo_path(os.environ.get("KAGGLEHUB_CACHE", "data/cache"))
SOURCE = repo_path(os.environ.get("MRI_DATA_SOURCE", str(
    cache / "datasets" / CONFIG["dataset"] / "versions" / str(CONFIG["version"]))))
print("Host:", platform.node(), "| Slurm job:", os.environ.get("SLURM_JOB_ID", "local"))
print("Repository:", REPO)
print("Audit:", AUDIT_DIR)
print("Source:", SOURCE)

## Generate an audit only if needed

The current Insomnia checkout already has the audit. To reproduce it elsewhere, activate the environment, install `requirements-data.txt`, and run the script **from a compute allocation**:

```bash
cd /path/to/brain-mri-vlm
export OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 MKL_NUM_THREADS=1
export KAGGLEHUB_CACHE="$PWD/data/cache"
python scripts/prepare_data.py --download --output data/prepared-v2-phash4-ncc98 --workers 2 --log-wandb
```

Use no more workers than allocated CPUs. Omit `--log-wandb` if not authenticated. Existing outputs are never overwritten; use a new output directory and update `AUDIT_DIR` above for another run. This notebook deliberately does not launch downloads or a full audit automatically.

In [ ]:
if not (AUDIT_DIR / "summary.json").is_file():
    raise FileNotFoundError(f"Generate an audit first, then set AUDIT_DIR: {AUDIT_DIR}")
if not (SOURCE / "Training").is_dir() or not (SOURCE / "Testing").is_dir():
    raise FileNotFoundError("Set SOURCE to the downloaded directory containing Training/ and Testing/.")

summary = json.loads((AUDIT_DIR / "summary.json").read_text())
protocol = {k: v for k, v in CONFIG.items() if k != "expected_source_sha256"}
if summary["protocol"] != protocol:
    raise ValueError("This audit uses another config. Regenerate it or review the matching config explicitly.")
expected = CONFIG.get("expected_source_sha256")
if expected and summary["source_inventory_sha256"] != expected:
    raise ValueError("Audit source identity differs from the pinned dataset.")
if summary["pipeline_sha256"] != hashlib.sha256((REPO / "scripts/prepare_data.py").read_bytes()).hexdigest():
    raise ValueError("Pipeline code changed since this audit was generated; regenerate the audit.")
for name, expected_hash in summary["output_sha256"].items():
    if Path(name).name != name:
        raise ValueError("Audit contains an invalid output filename.")
    if hashlib.sha256((AUDIT_DIR / name).read_bytes()).hexdigest() != expected_hash:
        raise ValueError(f"Audit file changed: {name}")

with (AUDIT_DIR / "duplicate_pairs.csv").open() as stream:
    pairs = list(csv.DictReader(stream))
with (AUDIT_DIR / "inventory.csv").open() as stream:
    inventory = {row["path"]: row for row in csv.DictReader(stream)}

def pair_id(pair):
    paths = sorted([pair["path_a"], pair["path_b"]])
    return hashlib.sha256(json.dumps(paths, separators=(",", ":")).encode()).hexdigest()[:16]

pairs_by_id = {pair_id(pair): pair for pair in pairs}
if len(pairs_by_id) != len(pairs):
    raise ValueError("Pair identifiers are not unique.")

REVIEW_FILE = REPO / "data/reviews" / f"pairs-{summary['protocol_sha256'][:12]}.csv"
print("Verified audit checksums; no manifests were changed.")
print("Candidate pairs:", len(pairs), "| Accepted:", sum(p["accepted"] == "True" for p in pairs))
print("Review notes will be saved locally to:", REVIEW_FILE)
print(json.dumps(summary["counts"], indent=2))

## Browse pairs

Filters: `accepted=True`, `False`, or `None` (both); `label` is a class name or `None`; `cross_split=True`, `False`, or `None`; `conflicts_only=True` limits to differing labels. Cross-split means **original Training vs Testing**, not final train vs validation.

`order="borderline"` starts near the correlation cutoff; `order="sample"` gives a reproducible mixed sample; `order="strongest"` starts with highest correlation. `start` and `count` paginate results. These views use the current audit's thresholds, not hardcoded replacements.

The displayed thumbnails apply EXIF orientation but keep each image's aspect ratio. Review image content rather than relying on either score alone.

In [ ]:
def select_pairs(accepted=True, label=None, cross_split=True,
                 conflicts_only=False, order="borderline"):
    if accepted is not None and not isinstance(accepted, bool):
        raise ValueError("accepted must be True, False, or None")
    if cross_split is not None and not isinstance(cross_split, bool):
        raise ValueError("cross_split must be True, False, or None")
    if label is not None and label not in CONFIG["labels"]:
        raise ValueError(f"label must be one of {CONFIG['labels']} or None")
    selected = [p for p in pairs
                if (accepted is None or (p["accepted"] == "True") == accepted)
                and (cross_split is None or (p["cross_source_split"] == "True") == cross_split)
                and (not conflicts_only or p["label_conflict"] == "True")
                and (label is None or inventory[p["path_a"]]["label"] == label
                     or inventory[p["path_b"]]["label"] == label)]
    cutoff = summary["protocol"]["min_correlation"]
    if order == "borderline":
        selected.sort(key=lambda p: (abs(float(p["correlation"]) - cutoff), pair_id(p)))
    elif order == "sample":
        selected.sort(key=lambda p: hashlib.sha256(
            f"{CONFIG['seed']}:{pair_id(p)}".encode()).hexdigest())
    elif order == "strongest":
        selected.sort(key=lambda p: (-float(p["correlation"]), pair_id(p)))
    else:
        raise ValueError("order must be borderline, sample, or strongest")
    return selected


def thumbnail(path):
    if path not in inventory:
        raise ValueError(f"Unknown audit image: {path}")
    image_path = (SOURCE / path).resolve()
    if not image_path.is_relative_to(SOURCE.resolve()):
        raise ValueError("Image path leaves source directory")
    raw = image_path.read_bytes()
    if hashlib.sha256(raw).hexdigest() != inventory[path]["sha256"]:
        raise ValueError(f"Source image changed since the audit: {path}")
    with Image.open(io.BytesIO(raw)) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        image.thumbnail((360, 360))
        buffer = io.BytesIO()
        image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode()


def show_pairs(accepted=True, start=0, count=3, label=None,
               cross_split=True, conflicts_only=False, order="borderline"):
    if start < 0 or not 1 <= count <= 20:
        raise ValueError("Use start >= 0 and count between 1 and 20")
    selected = select_pairs(accepted, label, cross_split, conflicts_only, order)
    page = selected[start:start + count]
    print(f"Showing {len(page)} of {len(selected)} matching candidates, starting at {start}.")
    for pair in page:
        ident = pair_id(pair)
        panels = []
        for key in ["path_a", "path_b"]:
            path = pair[key]
            panels.append(f'<div style="flex:1;min-width:220px"><p>{html.escape(path)}</p>'
                          f'<img alt="{html.escape(path)}" src="data:image/png;base64,{thumbnail(path)}" '
                          'style="max-width:100%;max-height:360px;object-fit:contain"></div>')
        status = "ACCEPTED" if pair["accepted"] == "True" else "REJECTED"
        display(HTML(f'<section style="border-top:1px solid #aaa;padding:12px 0">'
                     f'<b>{status} | pair ID: {ident}</b><p>'
                     f'pHash distance: {pair["hamming_distance"]} | '
                     f'Correlation: {float(pair["correlation"]):.6f} | '
                     f'Match type: {html.escape(pair["match_kind"])} | '
                     f'Label conflict: {pair["label_conflict"]}</p>'
                     '<div style="display:flex;flex-wrap:wrap;gap:16px">'
                     + "".join(panels) + '</div></section>'))
    return [pair_id(pair) for pair in page]

In [ ]:
# Accepted candidates: look for distinct images incorrectly grouped together.
accepted_ids = show_pairs(accepted=True, count=3, order="borderline")

In [ ]:
# Rejected candidates: look for transformed copies the strict rule missed.
rejected_ids = show_pairs(accepted=False, count=3, order="borderline")

In [ ]:
# Change these arguments to inspect another class, page, or label conflicts.
# show_pairs(accepted=True, label="notumor", start=3, count=3)
# show_pairs(accepted=False, label="notumor", order="sample", count=3)
# show_pairs(accepted=None, conflicts_only=True, cross_split=None, count=3)

## Record review notes locally

Use the pair ID shown above. Decisions are `duplicate`, `distinct`, or `uncertain`; add your name and a short reason. Saving notes **does not override** the pipeline's accepted flag or alter splits. The notes CSV includes the source and protocol identities and lives under ignored `data/reviews/`.

`save_review` updates the note for that pair in this review file, including a previous note by another reviewer. Use separate review files for independent reviewers. Share an agreed review file via W&B or a small reviewed Git addition later, rather than committing image outputs. Any resulting matching-policy changes belong in the script/config, followed by regeneration and a new data artifact.

In [ ]:
REVIEW_FIELDS = ["pair_id", "path_a", "path_b", "decision", "reviewer", "notes",
                 "source_inventory_sha256", "protocol_sha256"]

def load_reviews():
    if not REVIEW_FILE.exists():
        return {}
    with REVIEW_FILE.open() as stream:
        rows = list(csv.DictReader(stream))
    for row in rows:
        if (row["source_inventory_sha256"] != summary["source_inventory_sha256"]
                or row["protocol_sha256"] != summary["protocol_sha256"]):
            raise ValueError("Review file belongs to a different source/protocol")
    return {row["pair_id"]: row for row in rows}


def save_review(ident, decision, reviewer, notes=""):
    if ident not in pairs_by_id:
        raise ValueError("Unknown pair ID; copy one from the displayed audit")
    if decision not in {"duplicate", "distinct", "uncertain"}:
        raise ValueError("decision must be duplicate, distinct, or uncertain")
    if not reviewer.strip():
        raise ValueError("Enter your name or initials")
    pair = pairs_by_id[ident]
    reviews = load_reviews()
    reviews[ident] = dict(pair_id=ident, path_a=pair["path_a"], path_b=pair["path_b"],
                          decision=decision, reviewer=reviewer.strip(), notes=notes,
                          source_inventory_sha256=summary["source_inventory_sha256"],
                          protocol_sha256=summary["protocol_sha256"])
    REVIEW_FILE.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode="w", newline="", encoding="utf-8",
                                     dir=REVIEW_FILE.parent, delete=False) as stream:
        writer = csv.DictWriter(stream, fieldnames=REVIEW_FIELDS, lineterminator="\n")
        writer.writeheader()
        writer.writerows(reviews[key] for key in sorted(reviews))
        temporary = Path(stream.name)
    temporary.replace(REVIEW_FILE)
    print(f"Saved {len(reviews)} review notes to {REVIEW_FILE}")

# Example only: replace the ID, reviewer and reason, then uncomment.
# save_review(accepted_ids[0], "uncertain", "your initials", "Explain what needs another look")
print("Existing review notes:", len(load_reviews()))

## Before committing

Keep this notebook tracked **without outputs or execution counts**. Images, generated manifests and review notes stay under ignored `data/`. Clear notebook outputs before committing, for example from the repository root:

```bash
jupyter nbconvert --ClearOutputPreprocessor.enabled=True --inplace notebooks/data_preprocessing.ipynb
```

Next: review representative accepted/rejected pairs and conflicts, agree on the matching policy, regenerate if needed, and freeze a versioned manifest artifact before training.